In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BertConfig, BertForMaskedLM, BertTokenizer
from datasets import load_dataset

device="cuda"

In [2]:
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
config = BertConfig(vocab_size=bert_tokenizer.vocab_size, hidden_size=128, num_hidden_layers=2, 
                    num_attention_heads=4, intermediate_size=512, max_position_embeddings=128)
bert = BertForMaskedLM(config)

In [3]:
from datasets import load_dataset

def tokenize(example, tokenizer=bert_tokenizer):
    return tokenizer(example["text"], truncation=True, max_length=128, padding="max_length")

mlm_dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
mlm_dataset

Dataset({
    features: ['text'],
    num_rows: 36718
})

In [4]:
mlm_dataset = mlm_dataset.map(tokenize, batched=True)
mlm_dataset

Dataset({
    features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 36718
})

In [5]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

args = TrainingArguments(output_dir="./bert", num_train_epochs=5, per_device_train_batch_size=16)
mlm_collator = DataCollatorForLanguageModeling(bert_tokenizer)
trainer = Trainer(model=bert, args=args, train_dataset=mlm_dataset, data_collator=mlm_collator)
trainer_output = trainer.train()

Step,Training Loss
500,8.850999
1000,7.488951
1500,7.315510
2000,7.224983
2500,7.170404
3000,7.129379
3500,7.125734
4000,7.065012
4500,7.074075
5000,7.013620


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
from transformers import pipeline
torch.manual_seed(42)
fill_mask = pipeline("fill-mask", model=bert, tokenizer=bert_tokenizer)
top_predictions = fill_mask("The capital of [MASK] is Rome.")
top_predictions[0]

{'score': 0.042053379118442535,
 'token': 1010,
 'token_str': ',',
 'sequence': 'the capital of, is rome.'}

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
sentences = ["She's shopping", "She bought some shoes", "She's working"]
embeddings = model.encode(sentences, convert_to_tensor=True)
similarities = model.similarity(embeddings, embeddings)
similarities

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tensor([[1.0000, 0.6328, 0.5841],
        [0.6328, 1.0000, 0.3831],
        [0.5841, 0.3831, 1.0000]], device='cuda:0')

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "gpt2"
gpt2_tokenizer = AutoTokenizer.from_pretrained(model_id)
gpt2 = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto")

def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id, **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [9]:
prompt = "Scientists found a talking unicorn today. Here's the full story:"
torch.manual_seed(42)
generate(gpt2, gpt2_tokenizer, prompt, do_sample=True)

"Scientists found a talking unicorn today. Here's the full story:\n\nThere aren't lots of other unicorns and they have been making their way across the United States since at least the 1800s, but this year there weren't a solitary unicorn on the land. Today, there are around 1,000."

In [10]:
DEFAULT_TEMPLATE = "Capital city of France = Paris\nCapital city of {country} ="

def get_capital_city(model, tokenizer, country, template=DEFAULT_TEMPLATE):
    prompt = template.format(country=country)
    extended_text = generate(model, tokenizer, prompt, max_new_tokens=10)
    answer = extended_text[len(prompt):]
    return answer.strip().splitlines()[0].strip()

In [11]:
get_capital_city(gpt2, gpt2_tokenizer, "United Kingdom")

'London'

In [12]:
get_capital_city(gpt2, gpt2_tokenizer, "Mexico")

'Mexico City'

In [13]:
model_id = "mistralai/Mistral-7B-v0.3"
mistral7b_tokenizer = AutoTokenizer.from_pretrained(model_id)
mistral7b = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype="auto")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [14]:
prompt = "List some places I should visit in Paris."
generate(mistral7b, mistral7b_tokenizer, prompt)

'List some places I should visit in Paris.\n\nI’m going to Paris in a few weeks and I’m looking for some places to visit. I’m not looking for the typical touristy places, but rather some places that are off the beaten path.\n\nI’'

In [18]:
bob_introduction = """
Bob is an amazing chatbot. It knows everything and it's incredibly helpful.
"""

full_prompt=f"{bob_introduction}Me:{prompt}\nBob:"
extended_text = generate(mistral7b, mistral7b_tokenizer, full_prompt, max_new_tokens=100)
answer = extended_text[len(full_prompt):].strip()
answer.split("\nMe: ")[0]

"The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.\nMe:What's the best way to get around Paris?\nBob:The metro is the most efficient way to get around Paris.\nMe:What's the best time of year to visit Paris?\nBob:The best time to visit Paris is in the spring or fall, when the weather is mild and the crowds are smaller"

In [19]:
class BobTheChatbot:
    def __init__(self, model, tokenizer, introduction=bob_introduction, max_answer_length=10000):
        self.model = model
        self.tokenizer = tokenizer
        self.context = introduction
        self.max_answer_length = max_answer_length
    
    def chat(self, prompt):
        self.context += "\nMe: " + prompt + "\nBob:"
        context = self.context
        start_index = len(context)
        while True:
            extended = generate(self.model, self.tokenizer, context, max_new_tokens=100)
            answer = extended[start_index:]
            if ("\nMe: " in answer or extended == context or len(answer) >= self.max_answer_length):break
            context = extended
        answer = answer.split("\nMe: ")[0]
        self.context += answer
        return answer.strip()


In [20]:
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
bob.chat("List some places I should visit in paris")

"The Eiffel Tower, The Louvre, The Arc de Triomphe, The Notre Dame Cathedral, The Sacré-Cœur Basilica, The Palace of Versailles, The Champs-Élysées, The Musée d'Orsay, The Centre Pompidou, The Jardin des Tuileries, The Jardin du Luxembourg, The Père Lachaise Cemetery, The Montmartre, The Latin Quarter, The Marais, The Saint-Germain-des-Prés, The Bastille, The Canal Saint-Martin, The Place de la Concorde, The Place de la Bastille, The Place de la République, The Place des Vosges, The Place de la Nation, The Place de la Madeleine, The Place de l'Opéra, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place de la Bourse, The Place d

In [21]:
bob.chat("tell me more about the first place")

'The Eiffel Tower is a wrought iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower.'

In [22]:
bob.chat("And Rome?")

"Rome is the capital city and a special comune of Italy. Rome also serves as the capital of the Lazio region. With 2,872,800 residents in 1,285 km2 (496.1 sq mi), it is also the country's most populated comune."

In [ ]:
prompt = "The capital of Argentina is "
full_input = [prompt + "Buenos Aires", prompt + "Madrid"]
mistral7b_tokenizer.pad_token = 